# Load a model and chat

Choose one of our supported local models, load it, say hello, and send a follow-up. This uses the same shared code the future application will use. No RAG yet.

Run setup and [model_download.ipynb](model_download.ipynb) first. Select QAChat's `.venv` kernel. Run one cell at a time with Shift+Enter. This notebook never downloads anything; the loading cell puts the chosen model into memory.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "app_core/chat.py").is_file():
    root = root.parent
assert (root / "app_core/chat.py").is_file(), "Open this notebook from QAChat."
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
assert Path(sys.prefix).resolve() == (root / ".venv").resolve(), "Select QAChat's .venv kernel."

from app_core.models import list_models, model_home
from app_core.chat import load_model

print("Python:", sys.executable)

## 1. See your model choices

Use the same storage folder as the download notebook. `None` means QAChat's ignored `models/` folder, unless you set `QACHAT_MODEL_HOME`.

This lists our supported catalog, not every model on the internet. Choose a downloaded model whose engine works on your computer. File sizes are not the memory needed to run a model.

In [ ]:
MODEL_DIRECTORY = None
# For another folder: MODEL_DIRECTORY = Path("~/my-models")
print("Model storage:", model_home(MODEL_DIRECTORY))

for model in list_models(directory=MODEL_DIRECTORY):
    print(f"{model['model']}: {model['backend']}, {model['size_gb']} GB, {model['status']}")

## 2. Choose one model

For Apple Silicon: `qwen3-14b-mlx` or the larger `qwen3.8-27b-mlx`. For PyTorch on Mac, Windows, or Linux: `qwen3-0.6b-torch` (download it first).

`auto` uses the chosen model's engine. You may enter `mlx` or `torch` explicitly, but the engine must match the weight files. This does not convert models between formats.

In [ ]:
MODEL = "qwen3-14b-mlx"
BACKEND = "auto"
MAX_NEW_TOKENS = 128

## 3. Load it once

This checks the full contents of the local files, then loads the weights and tokenizer. It can take a little time and several GB of memory. Missing files produce an error with download instructions, never an automatic download.

Keep one model loaded at a time. This cell closes an earlier session before loading another. If memory is tight, close other model applications or choose a smaller model.

In [ ]:
if "chat" in globals():
    chat.close()

chat = load_model(MODEL, directory=MODEL_DIRECTORY, backend=BACKEND)
print(f"Loaded {chat.name} using {chat.backend} on {chat.device}.")

## 4. Say hello

Edit the message if you like. A token is a small piece of text; `MAX_NEW_TOKENS` limits the reply length, not the number of words. Wait for the reply to finish. The assertion checks that some text came back, not that the answer is correct.

In [ ]:
MESSAGE = "Hello! My name is Iok-ui. Greet me in one short sentence."
answer = chat.reply(MESSAGE, max_new_tokens=MAX_NEW_TOKENS)
assert isinstance(answer, str) and answer.strip(), "Expected a non-empty reply."
print(answer)

## 5. Continue the same conversation

The model receives your earlier messages and its replies. Change this question and rerun the cell to keep chatting. Replies may be wrong or cut short by the length limit.

History stays in this session's memory, not on disk. For this small example, input plus reply is limited to 4096 tokens. Call `chat.reset()` for a fresh conversation without reloading the model.

In [ ]:
MESSAGE = "What name did I just tell you?"
answer = chat.reply(MESSAGE, max_new_tokens=MAX_NEW_TOKENS)
print(answer)

## 6. Finish and release the model

Run this when finished or before switching models. It clears this session's history and releases its model memory; downloaded files remain on disk. To chat again after closing, rerun the loading cell.

The model weights and generated output do not belong in Git. Clear saved notebook outputs before committing.

In [ ]:
chat.close()
print("Chat closed. Model files remain available for next time.")